# Setup

In [1]:
import os
import numpy as np
import pandas as pd

os.environ["GCLOUD_PROJECT"] = "frog-id-ml"
from NatureLM.infer import Pipeline

from NatureLM.models import NatureLM
from huggingface_hub import notebook_login

/home/admin/.virtualenvs/python3.11-venv/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
2026-04-13 08:39:54.254800: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# Authenticate to HuggingFace
notebook_login()

In [5]:
# Download the model from HuggingFace
model = NatureLM.from_pretrained("EarthSpeciesProject/NatureLM-audio")
model = model.eval().to("cuda")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


trainable params: 27,262,976 || all params: 8,057,532,416 || trainable%: 0.3384


/home/admin/.virtualenvs/python3.11-venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
BertLMHeadModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are n

# Data Annotation for Frog Audio Recordings

1. For each unique recording ID, run the frog vs not-a-frog query over the original and source-separated audio files (assumed to all be in the same folder)
2. Take the audio file with the highest occurence of "frog" predictions to be the best candidate
3. Extract the detection windows that have been predicted as a frog and save as new audio files for training dataset

In [ ]:
import os
import pandas as pd
from pydub import AudioSegment
from tqdm import tqdm

from pathlib import Path
import re
from collections import defaultdict


# Define audio input directory (should contain source-separated audio files)
AUDIO_DIR = "/home/admin/julia-dev/data/random_test/cane_toad_preprocessed"

# Define audio output directory (for saving the positive detection windows)
OUTPUT_DIR = "/home/admin/julia-dev/data/random_test/cane_toad_annotated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [18]:
folder = Path(AUDIO_DIR)
audio_path_dict = defaultdict(list)

# Extract original and source-separated audio paths
for f in folder.iterdir():
    if f.is_file():
        base = re.sub(r"_source[0-3]$", "", f.stem)
        audio_path_dict[base].append(os.path.join(AUDIO_DIR, f))

for base, files in audio_path_dict.items():
    print("Recording ID:", base)
    for f in files:
        print("  ", f)

Recording ID: cane_toad_01
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_01_source1.wav
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_01_source0.wav
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_01_source2.wav
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_01_source3.wav
Recording ID: cane_toad_02
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_02_source0.wav
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_02_source2.wav
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_02_source3.wav
   /home/admin/julia-dev/data/random_test/cane_toad_preprocessed/cane_toad_02_source1.wav


In [19]:
query = "The objective is to classify the sound into one of the following categories: frog, birds, insects"

pipeline = Pipeline(model=model)

# Run the query over each capture ID
for capture_id, audio_paths in tqdm(audio_path_dict.items(), desc="Running query over capture IDs"):
    results = pipeline(audio_paths, query, window_length_seconds=5.0, hop_length_seconds=5.0)

    # Find the best candidate file based on max number of positive predictions
    frog_counts = [x.count("frog") for x in results]
    max_idx = frog_counts.index(max(frog_counts))

    best_file = audio_paths[max_idx]
    best_file_name = best_file.split('/')[-1][:-4]
    best_results = results[max_idx].split('\n')

    # Load best candidate file
    audio = AudioSegment.from_wav(best_file)
    audio = audio.set_channels(1)

    # Extract positive detection windows
    for line in best_results:
        if line == '':
            continue
        _, time_range, predicted_label = line.split('#')
        predicted_label = predicted_label[2:]
        if predicted_label != "frog":
            continue
        
        # Get start and end times
        start_time, end_time = time_range.split(' - ')
        start_time_s = int(start_time[:-4])
        end_time_s = int(end_time[:-4])
        start_time_ms, end_time_ms = start_time_s * 1000, end_time_s * 1000

        # Crop audio to detection window and save as separate file
        chunk = audio[start_time_ms:end_time_ms]
        output_path = os.path.join(OUTPUT_DIR, f"{best_file_name}_{start_time_s}s_{end_time_s}s.wav")
        chunk.export(output_path, format="wav")

        print(f"Saved audio window to: {output_path}")

Running query over capture IDs:   0%|          | 0/2 [00:00<?, ?it/s]/home/admin/.virtualenvs/python3.11-venv/lib/python3.11/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/admin/.virtualenvs/python3.11-venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/admin/.virtualenvs/python3.11-venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_i

#### Audio encoding cache hit ####


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


#### Audio encoding cache hit ####


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


#### Audio encoding cache hit ####


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Running query over capture IDs:  50%|█████     | 1/2 [00:03<00:03,  3.79s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saved audio window to: /home/admin/julia-dev/data/random_test/cane_toad_annotated/cane_toad_01_source2_10s_15s.wav
Saved audio window to: /home/admin/julia-dev/data/random_test/cane_toad_annotated/cane_toad_01_source2_15s_20s.wav


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Saved audio window to: /home/admin/julia-dev/data/random_test/cane_toad_annotated/cane_toad_02_source3_0s_5s.wav
Saved audio window to: /home/admin/julia-dev/data/random_test/cane_toad_annotated/cane_toad_02_source3_5s_10s.wav
Saved audio window to: /home/admin/julia-dev/data/random_test/cane_toad_annotated/cane_toad_02_source3_10s_15s.wav
Saved audio window to: /home/admin/julia-dev/data/random_test/cane_toad_annotated/cane_toad_02_source3_15s_20s.wav


# Ablation Study

## Ablation #1 - Remove frog vs not-a-frog filtering

In [ ]:
import os
import math
import pandas as pd
from pydub import AudioSegment
from tqdm import tqdm


# Load existing dataset
annotated_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio/audio_annotated"

# Extract unique filenames in dataset (ignoring time windows)
filenames = [x.split('.')[0] for x in os.listdir(annotated_audio_dir) if os.path.isfile(os.path.join(annotated_audio_dir, x))]
stems = ['_'.join(x.split('_')[:-2]) for x in filenames]
unique_stems = list(set(stems))

print(f'{len(unique_stems)} unique files')

In [ ]:
original_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio"
preprocessed_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio/audio_preprocessed"


# Load audio
output_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio/audio_chunked_no_filtering"
for filename in tqdm(unique_stems, desc="Chunking audio files"):
    filepath = (os.path.join(preprocessed_audio_dir, filename) if "source" in filename else os.path.join(original_audio_dir, filename)) + ".wav"
    audio = AudioSegment.from_wav(filepath)

    # Chunk into 5s windows
    duration_ms = len(audio)
    chunk_length_ms = 5 * 1000
    total_chunks = math.ceil(duration_ms / chunk_length_ms)

    # Export each chunk as an individual wav file
    for i in range(total_chunks):
        start_ms = i * chunk_length_ms
        end_ms = min((i + 1) * chunk_length_ms, duration_ms)
        chunk = audio[start_ms:end_ms]
        output_path = os.path.join(output_dir, f"{filename}_chunk{i}.wav")
        chunk.export(output_path, format="wav")

## Ablation #2 - Remove source separation

In [ ]:
import os
import math
import pandas as pd
from pydub import AudioSegment
from tqdm import tqdm


# Load existing dataset
annotated_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp/audio_29spp_annotated"

# Extract unique capture IDs in dataset (ignoring source ID and time windows)
capture_ids = [x.split('_')[0] for x in os.listdir(annotated_audio_dir) if os.path.isfile(os.path.join(annotated_audio_dir, x))]
unique_capture_ids = list(set(capture_ids))
print(f'{len(unique_capture_ids)} unique files')

In [ ]:
query = "The objective is to classify the sound into one of the following categories: frog, birds, insects"
pipeline = Pipeline(model=model)

original_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp"

audio_paths = [os.path.join(original_audio_dir, f'{x}.wav') for x in unique_capture_ids]

# Run the query over each capture ID
results = pipeline(audio_paths, query, window_length_seconds=5.0, hop_length_seconds=5.0)

results

In [ ]:
# Extract positive detection windows for each capture
output_dir = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp/audio_29spp_annotated_no_ss"

for i, result in enumerate(results):
    capture_id = unique_capture_ids[i]
    audio_path = audio_paths[i]
    audio = AudioSegment.from_wav(audio_path)

    for line in result.split('\n'):
        if line == '':
            continue
        _, time_range, predicted_label = line.split('#')
        predicted_label = predicted_label[2:]
        if predicted_label != "frog":
            continue
        
        # Get start and end times
        start_time, end_time = time_range.split(' - ')
        start_time_s = int(start_time[:-4])
        end_time_s = int(end_time[:-4])
        start_time_ms, end_time_ms = start_time_s * 1000, end_time_s * 1000

        # Crop audio to detection window and save as separate file
        chunk = audio[start_time_ms:end_time_ms]
        output_path = os.path.join(output_dir, f"{capture_id}_{start_time_s}s_{end_time_s}s.wav")
        chunk.export(output_path, format="wav")

        print(f"Saved audio window to: {output_path}")